# Acontext — Step-by-Step Demo

**Acontext** provides Agent Skills as a Memory Layer for production AI agents.

| Layer | What it does |
|-------|-------------|
| **Short-term** — *Store* | Session-scoped message storage with multi-format support |
| **Mid-term** — *Observe* | Automatic task extraction and progress tracking |
| **Long-term** — *Learn* | Persistent skills learned from past sessions |

This notebook covers the three core features in ~15 minutes.

## 0 · Setup

1. Go to [acontext.io](https://acontext.io) and claim your free credits
2. Grab your API key from the dashboard
3. Run the cell below

In [ ]:
# pip install -q acontext

import os, time
from acontext import AcontextClient

client = AcontextClient(
    api_key="sk-ac-xxx"
)
print("Connected:", client.ping())

Connected: pong


### Using claude code, cursor, codex...?

Run `npx skills add memodb-io/Acontext-dev-skill`

let your agent explain Acontext to you.

---

## 1 · Short-term Memory — *Store*

Every agent conversation lives in a **session**. You store messages in any LLM format (OpenAI, Anthropic, Gemini) and retrieve them in any format.

Think of it as the agent's **working memory** for the current conversation.

In [13]:
session = client.sessions.create(user="demo-user@example.com")
print(f"Session created: {session.id}")

conversation = [
    {"role": "user", "content": "Build me a landing page for my SaaS product."},
    {
        "role": "assistant",
        "content": (
            "Sure! Here's my plan:\n"
            "1. Use Next.js with Tailwind CSS\n"
            "2. Hero section with CTA\n"
            "3. Feature grid, pricing table, footer\n"
            "4. Deploy to Vercel"
        ),
    },
    {"role": "user", "content": "Looks good — use a dark theme and add testimonials."},
    {
        "role": "assistant",
        "content": "Got it. I'll add a testimonials carousel and switch to a dark color palette. Starting now...",
    },
    {"role": "user", "content": "Also add an FAQ section at the bottom."},
    {
        "role": "assistant",
        "content": (
            "Done! The landing page now includes:\n"
            "- Dark theme\n- Hero + CTA\n- Feature grid\n"
            "- Testimonials carousel\n- Pricing table\n"
            "- FAQ accordion\n- Footer with social links"
        ),
    },
]

for msg in conversation:
    client.sessions.store_message(session.id, blob=msg, format="openai")

print(f"Stored {len(conversation)} messages")

Session created: a56e0813-9741-48ca-9d07-9a7cc8a89ec1
Stored 6 messages


In [15]:
result = client.sessions.get_messages(session.id, format="anthropic")

for msg in result.items:
    content = msg["content"]
    print(f"[{msg['role']}] {content}")

[user] [{'text': 'Build me a landing page for my SaaS product.', 'type': 'text'}]
[assistant] [{'text': "Sure! Here's my plan:\n1. Use Next.js with Tailwind CSS\n2. Hero section with CTA\n3. Feature grid, pricing table, footer\n4. Deploy to Vercel", 'type': 'text'}]
[user] [{'text': 'Looks good — use a dark theme and add testimonials.', 'type': 'text'}]
[assistant] [{'text': "Got it. I'll add a testimonials carousel and switch to a dark color palette. Starting now...", 'type': 'text'}]
[user] [{'text': 'Also add an FAQ section at the bottom.', 'type': 'text'}]
[assistant] [{'text': 'Done! The landing page now includes:\n- Dark theme\n- Hero + CTA\n- Feature grid\n- Testimonials carousel\n- Pricing table\n- FAQ accordion\n- Footer with social links', 'type': 'text'}]


---

## 2 · Mid-term Memory — *Observe*

Acontext **automatically watches** conversations and extracts structured tasks — no manual tracking needed.

Each user request becomes a tracked task with a status (`pending` → `running` → `success` / `failed`), progress log, and captured user preferences.

In [16]:
print(client.sessions.get_session_summary(session.id))

<task id="1" description="Build me a landing page for my SaaS product.">
<progress>
1. Planned landing page structure: Hero section with CTA, Feature grid, Pricing table, Footer. Using Next.js with Tailwind CSS. To deploy to Vercel.
2. Added testimonials carousel and switched to dark color palette.
3. Added FAQ accordion section.
4. Completed landing page with: Dark theme, Hero + CTA, Feature grid, Testimonials carousel, Pricing table, FAQ accordion, Footer with social links.
</progress>
</task>


---

## 3 · Long-term Memory — *Learn*

Long-term memory lets your agent **learn from past sessions** and carry that knowledge forward. Learned information is stored as human-readable markdown *skill files*.

How it works:
1. Create a **learning space** — a container for accumulated knowledge
2. **Link a session** — Acontext analyzes the conversation and extracts durable facts
3. **Read skills** — retrieve learned knowledge to inject into future prompts

In [17]:
space = client.learning_spaces.create(user="demo-user@example.com")
print(f"Learning space created: {space.id}")

learn_session = client.sessions.create(user="demo-user@example.com")
print(f"Session for learning: {learn_session.id}")

# Attach session to learning space BEFORE storing messages
client.learning_spaces.learn(space.id, session_id=learn_session.id)
print("Session attached — now storing messages...")

learn_conversation = [
    {"role": "user", "content": "My name is Gus."},
    {"role": "assistant", "content": "Nice to meet you, Gus!"},
    {"role": "user", "content": "Today I'm working on projects with guys in NUS!"},
]

for msg in learn_conversation:
    client.sessions.store_message(learn_session.id, blob=msg, format="openai")

client.sessions.flush(learn_session.id)

Learning space created: c754e431-9fb2-4445-b068-8339763fd423
Session for learning: 009e6e7b-6979-4e9d-9162-9b5bc5302c52
Session attached — now storing messages...


{'status': 200, 'errmsg': ''}

In [18]:
result = client.learning_spaces.wait_for_learning(
    space.id,
    session_id=learn_session.id
)
print(f"Learning status: {result.status}")

skills = client.learning_spaces.list_skills(space.id)
for s in skills:
    client.skills.download(skill_id=s.id, path=os.path.join("skills", s.name))

Learning status: completed


---

## 4 · *Define your own Memory with Skills*

The default skills (`daily-logs`, `user-general-facts`) cover common cases, but you can **define your own memory structure** by writing a `SKILL.md` template.

For example, a `social-contacts` skill that maintains one markdown file per person the user mentions — with their role, company, and notes.

In [19]:
import zipfile, io
from acontext import FileUpload

skill_md = """---
name: "social-contacts"
description: "Track people the user interacts with — one file per person"
---
# Social Contacts

Maintain a file for each person the user mentions or interacts with.

## File Structure

One file per person, named [first-last].md (lowercase, hyphenated).
Each file has: # [Full Name], ## Basics (Relationship, Role, Company), ## Notes.

## Guidelines

- One person per file — do not combine people
- Only record information explicitly mentioned by the user
- Update existing entries when corrections are provided
- Keep entries factual and concise
"""

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as zf:
    zf.writestr("SKILL.md", skill_md)
buf.seek(0)

skill = client.skills.create(
    file=FileUpload(filename="social-contacts.zip", content=buf.read()),
)
print(f"Created skill: {skill.name} ({skill.id})")

Created skill: social-contacts (014a5a26-450a-4f90-ad38-cd6e0198b495)


In [20]:
custom_space = client.learning_spaces.create(user="demo-user@example.com")
client.learning_spaces.include_skill(custom_space.id, skill_id=skill.id)

print("Skills in this space:")
for s in client.learning_spaces.list_skills(custom_space.id):
    print(f"  {s.name}: {s.description}")

custom_session = client.sessions.create(user="demo-user@example.com")
client.learning_spaces.learn(custom_space.id, session_id=custom_session.id)

messages = [
    {"role": "user", "content": "Set up a meeting with Alice Chen — she's our new product manager at Acme Corp. She prefers morning slots."},
    {"role": "assistant", "content": "I've scheduled a meeting with Alice Chen for tomorrow at 10 AM."},
    {"role": "user", "content": "Also remind me that Bob Martinez from the DevOps team helped us fix the staging deploy issue last week."},
    {"role": "assistant", "content": "Noted — Bob Martinez from DevOps helped resolve the staging deployment issue."},
]

for msg in messages:
    client.sessions.store_message(custom_session.id, blob=msg)
client.sessions.flush(custom_session.id)

client.learning_spaces.wait_for_learning(custom_space.id, session_id=custom_session.id)
print("\nLearning complete — downloading skills...")

for s in client.learning_spaces.list_skills(custom_space.id):
    client.skills.download(skill_id=s.id, path=os.path.join("skills", s.name))

Skills in this space:
  social-contacts: Track people the user interacts with — one file per person
  daily-logs: Track daily activity logs and summaries for the user. TRIGGER BY: read/edit user memory
  user-general-facts: Capture and organize general facts about the user by topic

Learning complete — downloading skills...


---

## What else can you do?

Features not covered above — explore these on your own:

- **Context engineering**
- **Multi-modal, multi-provider**
- **Filesystem**
- **Sandboxes**
- **Dashboard**
- ...

## Go to [Acontext.io](https://acontext.io)!
